# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Data Joins and JSON columns** </center>
---
**Profesor**: Pablo Camarillo Ramirez

# Create SparkSession

In [1]:
from spark_utils import SparkUtils

# Create Spark session
spark_url = "spark://spark-master:7077"
app_name = "Example - Extract info from JSON"

su =SparkUtils(spark_url, app_name)
su

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/26 01:16:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<SparkUtils: Example - Extract info from JSON>

# Manipulating JSON Columns

In [2]:
json_schema = SparkUtils.generate_schema([("id", "int"), ("json_col", "string")])
json_data = [
    (1, '{"name": "Alice", "age": 25, "payments": [34, 433, 54], "address": {"city": "New York", "zip": "10001"}}'),
    (2, '{"name": "Bob", "age": 30, "address": {"city": "Los Angeles", "zip": "90001"}}'),
    (3, '{"name": "Charlie", "age": 35, "address": {"city": "Chicago", "zip": "60601"}}')
]

df_json = su._spark.createDataFrame(json_data, json_schema)
df_json.show(truncate=False)


+---+--------------------------------------------------------------------------------------------------------+
|id |json_col                                                                                                |
+---+--------------------------------------------------------------------------------------------------------+
|1  |{"name": "Alice", "age": 25, "payments": [34, 433, 54], "address": {"city": "New York", "zip": "10001"}}|
|2  |{"name": "Bob", "age": 30, "address": {"city": "Los Angeles", "zip": "90001"}}                          |
|3  |{"name": "Charlie", "age": 35, "address": {"city": "Chicago", "zip": "60601"}}                          |
+---+--------------------------------------------------------------------------------------------------------+



### Extract a JSON column with get_json_object function

In [3]:
from pyspark.sql.functions import get_json_object
df_json = df_json.withColumn("name", get_json_object(df_json.json_col, "$.name"))
df_json.show()

+---+--------------------+-------+
| id|            json_col|   name|
+---+--------------------+-------+
|  1|{"name": "Alice",...|  Alice|
|  2|{"name": "Bob", "...|    Bob|
|  3|{"name": "Charlie...|Charlie|
+---+--------------------+-------+



In [4]:
df_json = df_json.withColumn("age", get_json_object(df_json.json_col, "$.age"))
df_json.show()

+---+--------------------+-------+---+
| id|            json_col|   name|age|
+---+--------------------+-------+---+
|  1|{"name": "Alice",...|  Alice| 25|
|  2|{"name": "Bob", "...|    Bob| 30|
|  3|{"name": "Charlie...|Charlie| 35|
+---+--------------------+-------+---+



In [5]:
df_json = df_json.withColumn("city", get_json_object(df_json.json_col, "$.address.city"))
df_json.show()

+---+--------------------+-------+---+-----------+
| id|            json_col|   name|age|       city|
+---+--------------------+-------+---+-----------+
|  1|{"name": "Alice",...|  Alice| 25|   New York|
|  2|{"name": "Bob", "...|    Bob| 30|Los Angeles|
|  3|{"name": "Charlie...|Charlie| 35|    Chicago|
+---+--------------------+-------+---+-----------+



In [6]:
df_json = df_json.withColumn("1st_payment", get_json_object(df_json.json_col, "$.payments[0]"))
df_json.show()

+---+--------------------+-------+---+-----------+-----------+
| id|            json_col|   name|age|       city|1st_payment|
+---+--------------------+-------+---+-----------+-----------+
|  1|{"name": "Alice",...|  Alice| 25|   New York|         34|
|  2|{"name": "Bob", "...|    Bob| 30|Los Angeles|       NULL|
|  3|{"name": "Charlie...|Charlie| 35|    Chicago|       NULL|
+---+--------------------+-------+---+-----------+-----------+



### Extact a JSON column with from_json

In [7]:
from pyspark.sql.functions import from_json
# Deine the schema of the JSON object
people_schema = SparkUtils.generate_schema([("name", "string"),
                                            ("age", "int"),
                                            ("payments", "array_int"),
                                            ("address", "struct")])
df_parsed = df_json.withColumn("parsed", from_json(df_json.json_col, people_schema))
df_parsed.printSchema()
df_parsed.show(truncate=False)
df_parsed.toPandas()

root
 |-- id: integer (nullable = true)
 |-- json_col: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: string (nullable = true)
 |-- city: string (nullable = true)
 |-- 1st_payment: string (nullable = true)
 |-- parsed: struct (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- age: integer (nullable = true)
 |    |-- payments: array (nullable = true)
 |    |    |-- element: integer (containsNull = true)
 |    |-- address: struct (nullable = true)

+---+--------------------------------------------------------------------------------------------------------+-------+---+-----------+-----------+------------------------------+
|id |json_col                                                                                                |name   |age|city       |1st_payment|parsed                        |
+---+--------------------------------------------------------------------------------------------------------+-------+---+-----------+-----------+-----

,id,json_col,name,age,city,1st_payment,parsed
0,1,"{""name"": ""Alice"", ""age"": 25, ""payments"": [34, ...",Alice,25,New York,34,"(Alice, 25, [34, 433, 54], ())"
1,2,"{""name"": ""Bob"", ""age"": 30, ""address"": {""city"":...",Bob,30,Los Angeles,None,"(Bob, 30, None, ())"
2,3,"{""name"": ""Charlie"", ""age"": 35, ""address"": {""ci...",Charlie,35,Chicago,None,"(Charlie, 35, None, ())"


In [8]:
from pyspark.sql.functions import col
df_parsed.select(col("parsed.name"), col("parsed.payments").getItem(1)).show()

+-------+------------------+
|   name|parsed.payments[1]|
+-------+------------------+
|  Alice|               433|
|    Bob|              NULL|
|Charlie|              NULL|
+-------+------------------+



## Data Join

In [9]:
df_a = su._spark.createDataFrame([(1, "Alice"), (2, "Bob")],
                              ["id", "name"])
df_b = su._spark.createDataFrame([(3, "Charlie"), (4, "David")],
                              ["id", "name"])
result = df_a.union(df_b)
result.show()

+---+-------+
| id|   name|
+---+-------+
|  1|  Alice|
|  2|    Bob|
|  3|Charlie|
|  4|  David|
+---+-------+



### Union w/o duplicates

In [10]:
df_a = su._spark.createDataFrame([(1, "Alice"), (2, "Bob")],
                              ["id", "name"])
df_b = su._spark.createDataFrame([(1, "Alice"), (4, "David")],
                              ["id", "name"])
df_a.union(df_b).distinct().show()

[Stage 17:===========================================>              (3 + 1) / 4]

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
|  4|David|
+---+-----+



### Union with Mismatched Schemas

In [11]:
df_a = su._spark.createDataFrame([(1, "Alice")], ["id", "name"])
df_b = su._spark.createDataFrame([("Bob", 2)], ["name", "id"])
result = df_a.unionByName(df_b)
result.show()

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+



### Union by Name with missing columns

In [12]:
df_a = su._spark.createDataFrame([(1, "Alice", "NY")], ["id", "name", "city"])
df_a.show()
df_b = su._spark.createDataFrame([(2, "Bob")], ["id", "name"])
df_b.show()
result = df_a.unionByName(df_b, allowMissingColumns=True)
result.show()

+---+-----+----+
| id| name|city|
+---+-----+----+
|  1|Alice|  NY|
+---+-----+----+

+---+----+
| id|name|
+---+----+
|  2| Bob|
+---+----+

+---+-----+----+
| id| name|city|
+---+-----+----+
|  1|Alice|  NY|
|  2|  Bob|NULL|
+---+-----+----+



## Left Join

### Datasets

In [13]:
book_data = [
    ("Game of thrones", 400, 1),
    ("Spark", 500, 2),
    ("Kafka", 300, 3),
    ("Java", 350, 5)
]
df_books = su._spark.createDataFrame(book_data, ["book_name", "cost", "writer_id"])

writer_data = [
    ("George R.R. Martin", 1),
    ("Zaharia", 2),
    ("Neha", 3),
    ("James", 4)
]
df_writers = su._spark.createDataFrame(writer_data, ["writer_name", "writer_id"])

df_books.show()
df_writers.show()

+---------------+----+---------+
|      book_name|cost|writer_id|
+---------------+----+---------+
|Game of thrones| 400|        1|
|          Spark| 500|        2|
|          Kafka| 300|        3|
|           Java| 350|        5|
+---------------+----+---------+

+------------------+---------+
|       writer_name|writer_id|
+------------------+---------+
|George R.R. Martin|        1|
|           Zaharia|        2|
|              Neha|        3|
|             James|        4|
+------------------+---------+



In [14]:
result = df_books.join(df_writers, 
      df_books["writer_id"] == df_writers["writer_id"], "left")
result.show()

[Stage 33:=============================>                            (1 + 1) / 2]

+---------------+----+---------+------------------+---------+
|      book_name|cost|writer_id|       writer_name|writer_id|
+---------------+----+---------+------------------+---------+
|Game of thrones| 400|        1|George R.R. Martin|        1|
|          Spark| 500|        2|           Zaharia|        2|
|           Java| 350|        5|              NULL|     NULL|
|          Kafka| 300|        3|              Neha|        3|
+---------------+----+---------+------------------+---------+



In [15]:
result = df_books.join(df_writers, on="writer_id", how="left")
result.show()

+---------+---------------+----+------------------+
|writer_id|      book_name|cost|       writer_name|
+---------+---------------+----+------------------+
|        1|Game of thrones| 400|George R.R. Martin|
|        2|          Spark| 500|           Zaharia|
|        5|           Java| 350|              NULL|
|        3|          Kafka| 300|              Neha|
+---------+---------------+----+------------------+



## Right join

In [16]:
result = df_books.join(df_writers, df_books["writer_id"] == df_writers["writer_id"], "right")
result.show()

+---------------+----+---------+------------------+---------+
|      book_name|cost|writer_id|       writer_name|writer_id|
+---------------+----+---------+------------------+---------+
|Game of thrones| 400|        1|George R.R. Martin|        1|
|          Spark| 500|        2|           Zaharia|        2|
|          Kafka| 300|        3|              Neha|        3|
|           NULL|NULL|     NULL|             James|        4|
+---------------+----+---------+------------------+---------+



In [17]:
result = df_books.join(df_writers, on="writer_id", how="right")
result.show()

+---------+---------------+----+------------------+
|writer_id|      book_name|cost|       writer_name|
+---------+---------------+----+------------------+
|        1|Game of thrones| 400|George R.R. Martin|
|        2|          Spark| 500|           Zaharia|
|        3|          Kafka| 300|              Neha|
|        4|           NULL|NULL|             James|
+---------+---------------+----+------------------+



### Inner Join

In [18]:
df_books.join(df_writers, on="writer_id").show()

+---------+---------------+----+------------------+
|writer_id|      book_name|cost|       writer_name|
+---------+---------------+----+------------------+
|        1|Game of thrones| 400|George R.R. Martin|
|        2|          Spark| 500|           Zaharia|
|        3|          Kafka| 300|              Neha|
+---------+---------------+----+------------------+



In [19]:
su._spark.stop()